# Демонстрация как разные функции потерь реагируют на выборсы


Мы сравним две модели:

- одну, которая минимизирует MSE, — чувствительную к выбросам;
- другую — с функцией потерь MAE, которая устойчивее к шуму и аномалиям.

Вы увидите на практике, как выбор лосса влияет на устойчивость и характер предсказаний модели, — и научитесь осознанно подходить к этому выбору в своих проектах.

## Подготовка данных с выбросами


Сначала сгенерируем синтетическую линейную зависимость с шумом с помощью make_regression.

Затем исказим часть целевой переменной y, умножив некоторые значения на 5, — это позволит сымитировать выбросы.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_regression

# Генерация базовых данных
X, y = make_regression(n_samples=200,
                       n_features=1,
                       noise=50,
                       random_state=42)

# Преобразуем в DataFrame для удобства
X = pd.DataFrame(X, columns=['feature'])
y = pd.Series(y, name='target')

# Сохраним «чистую» версию y для сравнения
y_clean = y.copy()

# Превратим в выбросы 5% объектов
n_outliers = int(0.05 * len(y))
outlier_indices = np.random.RandomState(42).choice(y.index, size=n_outliers, replace=False)
y.loc[outlier_indices] *= 5  # Увеличим значения в 5 раз

## Формулировка задачи


Наша цель — сравнить поведение двух линейных моделей, обученных с разными функциями потерь:
- Модель 1 будет обучаться с функцией потерь MSE (среднеквадратичная ошибка).
- Модель 2 — с функцией потерь MAE (средняя абсолютная ошибка).

План действий:
- Выполним разбиение на train / val / test.
- Проведём масштабирование признаков с помощью реализованной в прошлом уроке функции scale_data().
- Обучим две линейные модели на одних и тех же данных. В качестве модели будем использовать SGDRegressor — он позволяет легко переключаться между разными лоссами.
- Получим предсказания на train, val и test.
- Рассчитаем метрики RMSE, MAPE и R² для обеих моделей с помощью функции calculate_metrics() из прошлого урока.

## Обучение моделей

In [3]:
# Импорт библиотек
from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score

# --- ФУНКЦИИ ---
# Масштабирование
def scale_data(X_train, X_val, X_test, method='standard'):
    if method == 'standard':
        mean = X_train.mean()
        std = X_train.std()
        X_train_scaled = (X_train - mean) / std
        X_val_scaled = (X_val - mean) / std
        X_test_scaled = (X_test - mean) / std
    elif method == 'minmax':
        min_val = X_train.min()
        max_val = X_train.max()
        X_train_scaled = (X_train - min_val) / (max_val - min_val)
        X_val_scaled = (X_val - min_val) / (max_val - min_val)
        X_test_scaled = (X_test - min_val) / (max_val - min_val)
    else:
        raise ValueError("Неверный метод масштабирования. Используйте 'standard' или 'minmax'.")
    return X_train_scaled, X_val_scaled, X_test_scaled

# --- ПОДГОТОВКА ДАННЫХ ---
# Генерация базовых данных
X, y = make_regression(n_samples=200,
                       n_features=1,
                       noise=50,
                       random_state=42)

# Преобразуем в DataFrame для удобства
X = pd.DataFrame(X, columns=['feature'])
y = pd.Series(y, name='target')

# Превратим в выбросы 5% объектов
n_outliers = int(0.05 * len(y))
outlier_indices = np.random.RandomState(42).choice(y.index, size=n_outliers, replace=False)
y.loc[outlier_indices] *= 5  # Увеличим значения в 5 раз

# Выделите на train_val и test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=True,
    random_state=42
  )  # напишите ваш код здесь

# Выделите на train и val
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.25,
    shuffle=True,
    random_state=42
    ) # напишите ваш код здесь

# Масштабирование
X_train_scaled, X_val_scaled, X_test_scaled = scale_data(X_train, X_val, X_test, method='standard') # ваш код здесь

# --- ОБУЧЕНИЕ МОДЕЛЕЙ ---
# Модель 1 — MSE
model_mse = SGDRegressor(loss='squared_error', random_state=42) # напишите ваш код здесь
# Обучение модели: напишите ваш код здесь
model_mse.fit(X_train_scaled, y_train)

# Модель 2 — MAE
model_mae = SGDRegressor(loss='epsilon_insensitive', epsilon=0.0, random_state=42) # напишите ваш код здесь
# Обучение модели: напишите ваш код здесь
model_mae.fit(X_train_scaled, y_train)

# --- ПРЕДСКАЗАНИЯ ---
y_train_pred_mse = model_mse.predict(X_train_scaled) # напишите ваш код здесь
y_val_pred_mse = model_mse.predict(X_val_scaled) # напишите ваш код здесь
y_test_pred_mse = model_mse.predict(X_test_scaled) # напишите ваш код здесь

y_train_pred_mae = model_mae.predict(X_train_scaled) # напишите ваш код здесь
y_val_pred_mae = model_mae.predict(X_val_scaled) # напишите ваш код здесь
y_test_pred_mae = model_mae.predict(X_test_scaled) # напишите ваш код здесь

# Вывод средних предсказаний
print(round(y_test_pred_mse.mean(), 2))
print(round(y_test_pred_mae.mean(), 2))

-9.18
-7.27


c:\Projects\yandex-practicum\.venv\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:1612: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


## Сравнение моделей с разными лоссами

После того как обе модели обучены, пора проанализировать результаты и понять, как выбор лосса влияет на поведение модели. Сравним модели по трём ключевым аспектам:

- Значения коэффициентов моделей — чтобы понять, как каждая модель учитывает признаки.
- Метрики качества на обучающей и валидационной выборках — чтобы оценить, насколько модели устойчивы и хорошо обобщают данные.
- Чувствительность к выбросам — посмотрим, насколько сильно каждая модель «перекосилась» из-за шума в данных.

Этот анализ поможет вам осознанно выбирать функцию потерь в задачах, где в данных возможны выбросы и шум.

### Сравнение коэффициентов


In [4]:
# Вывод значений коэффициентов моделей 
print("Коэффициенты модели с MSE:")
print("Константа (intercept):", round(model_mse.intercept_[0], 3))
print("Вес:", np.round(model_mse.coef_, 3))

print("\nКоэффициенты модели с MAE:")
print("Константа (intercept):", round(model_mae.intercept_[0], 3))
print("Вес:", np.round(model_mae.coef_, 3))

Коэффициенты модели с MSE:
Константа (intercept): 3.945
Вес: [73.036]

Коэффициенты модели с MAE:
Константа (intercept): 1.173
Вес: [47.019]


Коэффициенты и константы у моделей с разными функциями потерь различаются существенно — это наглядно показывает, как лосс влияет на поведение модели:

- MSE-модель выдала высокий коэффициент (≈73) и большую константу (≈3,9). Это связано с тем, что функция потерь MSE усиливает влияние выбросов — модель старается подстроиться под крупные отклонения, что приводит к завышенным весам.
- MAE-модель получила более скромный коэффициент (≈47) и меньшую константу (≈1,2). MAE менее чувствительна к экстремальным значениям, поэтому веса остаются более устойчивыми.

Можно сделать следующие выводы о влиянии выбора функции потерь на значения коэффициентов:

- Различия в коэффициентах на одних и тех же данных говорят о том, что выбор функции потерь напрямую влияет на характер модели.
- MAE-модель устойчивее к шуму и даёт более стабильные веса, в то время как MSE-модель может резко менять коэффициенты даже из-за одного выброса в данных.

### Сравнение метрик качества



In [6]:
# Расчёт метрик
def calculate_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {'RMSE': round(rmse, 2),
        'MAPE': round(mape * 100, 2),
        'R2': round(r2, 3)}

# --- МЕТРИКИ ---
mse_train_metrics = calculate_metrics(y_train, y_train_pred_mse)
mse_val_metrics = calculate_metrics(y_val, y_val_pred_mse)

mae_train_metrics = calculate_metrics(y_train, y_train_pred_mae)
mae_val_metrics = calculate_metrics(y_val, y_val_pred_mae)

print("MSE-модель (train):", mse_train_metrics)
print("MSE-модель (val):", mse_val_metrics)
print()
print("MAE-модель (train):", mae_train_metrics)
print("MAE-модель (val):", mae_val_metrics)

MSE-модель (train): {'RMSE': np.float64(48.28), 'MAPE': 293.66, 'R2': 0.694}
MSE-модель (val): {'RMSE': np.float64(55.37), 'MAPE': 92.89, 'R2': 0.722}

MAE-модель (train): {'RMSE': np.float64(54.88), 'MAPE': 211.78, 'R2': 0.605}
MAE-модель (val): {'RMSE': np.float64(68.36), 'MAPE': 88.01, 'R2': 0.576}


MSE-модель показывает более высокое качество на train (RMSE ≈ 48, R² ≈ 0,69) и в целом сохраняет его на валидации (RMSE ≈ 55, R² ≈ 0,72). Улучшение R² при переходе к val говорит о том, что модель не переобучилась, а, наоборот, смогла неплохо обобщить закономерности.

MAE-модель чуть хуже справляется на train (RMSE ≈ 55, R² ≈ 0,61) и на валидации тоже даёт более слабые результаты (RMSE ≈ 68, R² ≈ 0,58). Однако по MAPE она выглядит предпочтительнее (88% против 93% у MSE), что отражает её устойчивость к крупным выбросам.

Итак, MSE-модель даёт более высокое качество в целом и лучше объясняет дисперсию данных, а MAE-модель ведёт себя устойчивее при выбросах и показывает меньшую процентную ошибку. Выбор между ними зависит от того, что важнее в задаче — максимальная точность на большинстве точек или снижение влияния редких экстремальных значений.

MSE, будучи чувствительной к большим ошибкам, часто даёт высокое качество на обучении на метриках типа RMSE и R², но сильнее зависит от выбросов.

MAE менее чувствительна к шуму и выбросам, зато иногда проигрывает в общей точности на чистых данных.

### Чувствительность к выбросам


В уроке есть график по которому видно

MSE-модель больше старается приблизить свои предсказания к выбросам. Это приводит к тому, что она смещает предсказания даже для «нормальных» точек. В результате — меньшая точность внутри основной группы данных.

MAE-модель больше «игнорирует» выбросы и сглаживает предсказания, стараясь сохранить общий тренд. Поэтому она лучше предсказывает типичные наблюдения, не отвлекаясь на шум.

**Основные выводы о чувствительности функций потерь к выбросам:**
- MSE-модель склонна к переобучению на выбросах — предсказания «расползаются» ради минимизации больших ошибок.
- MAE-модель устойчива к шуму — она «держится середины» и сохраняет устойчивость к редким экстремальным значениям, что даёт лучшую обобщающую способность.

## Финальная оценка качества


In [8]:
# Относительное изменение метрик
def compare_metrics(train_metrics, val_metrics):
    for metric in train_metrics:
        change = (val_metrics[metric] - train_metrics[metric]) / train_metrics[metric] * 100
        direction = "изменилась" if metric != 'R2' else "изменился"
        print(f"{metric} на val {direction} по сравнению с {metric} на train на {change:.2f}%")

# --- МЕТРИКИ ---
mae_test_metrics = calculate_metrics(y_test, y_test_pred_mae)
mse_test_metrics = calculate_metrics(y_test, y_test_pred_mse)

print("MAE-модель (test):", mae_test_metrics)
print("MSE-модель (test):", mse_test_metrics)

MAE-модель (test): {'RMSE': np.float64(197.09), 'MAPE': 80.87, 'R2': 0.261}
MSE-модель (test): {'RMSE': np.float64(182.66), 'MAPE': 84.77, 'R2': 0.365}


Проанализируем полученные результаты:

- RMSE = 197,09. Средняя ошибка предсказания — порядка 200 единиц. Для представленных данных это высокая ошибка.
- MAPE = 81%. В среднем модель ошибается почти на 81% от истинного значения. Это плохой результат — предсказания сильно расходятся с реальностью, особенно для небольших значений таргета. Проблема кроется в данных — make_regression генерирует близкие к нулю таргеты, из-за чего MAPE теряет смысл.
- R² = 0,261. Модель объясняет всего 26,1% дисперсии целевой переменной. То есть модель слабо улавливает зависимость между признаками и целевой переменной и работает ненамного лучше тривиального предсказания среднего.

Обратите внимание: на тестовой выборке качество модели снизилось. Это не ошибка и не «поломка» алгоритма — просто в тестовых данных оказалось несколько выбросов, чьё влияние является непропорционально сильным. Такое поведение естественно при работе с шумными данными. Именно поэтому вы видите, что даже MAE-модель не всегда справляется идеально, хотя и остаётся более устойчивой, чем MSE.

Главный вывод такой: если в данных много шума, даже устойчивые модели не покажут высокого качества. В реальных задачах это сигнал, что данные стоит дополнительно очищать, преобразовывать или применять более продвинутые методы.

Итак, модель слабо справляется с задачей. Ошибки высокие, а R² показывает, что фактически модель почти не объясняет новые данные. Главная причина кроется в выбросах, с которыми ни MAE-модель, ни MSE-модель не научились хорошо справляться.